In [ ]:
"""
案例:
演示 dropout 正则化.

    dropout正则化解释:
    概述:
        让神经元以 p概率 随机失活, 防止模型训练时过度依赖某些特征.
        p的值一般是: 0.2 ~ 0.5, 如果是简单模型可以小一点, 如果是复杂模型可以大一点.
    细节:
        1. 未被失活的数据(神经元)会被缩放, 缩放比例是: 1 / (1 - p)
        2. dropout一般应用于 激活层之后, 即: 加权求和 -> 激活函数 -> dropout
        3. dropout只应用于 训练阶段, 在测试阶段失效.
            设置模型状态的两个函数如下:
                model.train()       ->  训练阶段
                model.eval()        ->  测试阶段
            
"""

In [32]:
import torch
import torch.nn as nn

In [34]:
# Dropout正则化
# 创建固定的随机种子 设置之后randint的值会固定
torch.manual_seed(10)
# 创建随即失活层 如果是类的写法 需要创建到init方法里作为属性调用
dropout = nn.Dropout(p = 0.4)
input = torch.randint(1,31,(1,6),dtype=torch.float)
# 输入特征数量 会随机正态 部分数据会负数
layer = nn.Linear(6,6)
y = layer(input)
nn.init.uniform_(y,1,2)
print(f"y:\n{y}")
y = torch.relu(y)
print(f"输入是：\n{input}")
print(f"未失活的输出结果：\n{y}")
y = dropout(y)
print(f"失活的输出结果：\n{y}")

y:
tensor([[1.1093, 1.4771, 1.1076, 1.9829, 1.1483, 1.5956]],
       grad_fn=<AddmmBackward0>)
输入是：
tensor([[18., 16., 13.,  8., 13.,  6.]])
未失活的输出结果：
tensor([[1.1093, 1.4771, 1.1076, 1.9829, 1.1483, 1.5956]],
       grad_fn=<ReluBackward0>)
失活的输出结果：
tensor([[0.0000, 2.4618, 0.0000, 3.3048, 1.9139, 0.0000]],
       grad_fn=<MulBackward0>)


In [ ]:
"""
案例:
    演示批量归一正则化.

批量归一正则化解释:
    概述:
        当数据量比较大的时候, 我们会分批次对数据进行训练, 但是如果批次之间的差异较大,
        导致 模型参数 需要大范围调整, 以便适应不同批次的"规律".
        这样就会导致: 模型的收敛速度变慢, 甚至可能出现"梯度震荡, 梯度爆炸"...
    解决思路:
        1. 对批次数据进行 标准化操作, 相当于减少了 批次数据之间的 差异性, 让数据更稳定.
        2. 弊端: 可能会丢失某些特征信息, 导致预测结果偏差, 所以引入了 λ(缩放, 当做: 权重) 和 β(平移, 当做: 偏置)
            f(x) = λ * (标准化后的x) + β

    批量归一化层, 常用的 3种方式:
        BatchNorm1d：主要应用于全连接层或处理一维数据的网络，例如文本处理。它接收形状为 (N, num_features) 的张量作为输入。
        BatchNorm2d：主要应用于卷积神经网络，处理二维图像数据或特征图。它接收形状为 (N, C, H, W) 的张量作为输入。
        BatchNorm3d：主要用于三维卷积神经网络 (3D CNN)，处理三维数据，例如视频或医学图像。它接收形状为 (N, C, D, H, W) 的张量作为输入。
"""

In [ ]:
# 批量归一化 -> 实际是批量的标准化
# 作用就是把输入/输出的变量变成0为中心范围是(-1,1)或(-3,3)之间使得激活函数的效果更好
# 二维的批量归一化
model = nn.BatchNorm2d(2, momentum=0.1, affine=True)
input = torch.randn(size=(1,2,3,4), dtype=torch.float)
output = model(input)
print(input)
print(output.size())
print(model.weight)
print(model.bias)


tensor([[[[-1.3013,  0.9827,  0.1463, -0.9752],
          [-1.1536,  0.0301, -0.1444,  0.8446],
          [ 0.2281,  0.6740, -0.2164,  0.7698]],

         [[-1.4489, -0.5613,  0.3234, -0.6428],
          [-1.0959, -0.3056, -0.0540,  0.8998],
          [-0.4482,  1.0605, -1.3841, -1.3646]]]])
torch.Size([1, 2, 3, 4])
Parameter containing:
tensor([1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0.], requires_grad=True)


In [47]:
torch.manual_seed(0)
x = torch.randint(5,20,(40,50),dtype=torch.float)
# 创建神经元
linear = nn.Linear(50,40)
# 加权求和
y = linear(x)
print(f"x:\n{x}")
print(y.size())
print(f"y:\n{y}")
# 一维的批量归一化
model = nn.BatchNorm1d(40,momentum=0.1,affine=True)
# 激活函数
result = model(y)
print(f"result:\n{result}")

x:
tensor([[19., 14., 13.,  ..., 15.,  7.,  6.],
        [15., 19., 13.,  ..., 15.,  7., 12.],
        [10., 11.,  6.,  ..., 19., 15.,  9.],
        ...,
        [ 5., 19., 15.,  ...,  5., 17., 19.],
        [10.,  8., 19.,  ..., 15., 10., 16.],
        [ 6., 11., 16.,  ..., 10., 14., 16.]])
torch.Size([40, 40])
y:
tensor([[ 2.8641, -1.6885,  2.9444,  ..., 16.7845, -3.4781, -5.3677],
        [ 6.9326, -7.1690, -1.1716,  ..., 14.6560, -1.3133, -1.8849],
        [-0.5454,  0.4026, -1.5763,  ...,  6.5578,  4.2498, -1.8403],
        ...,
        [ 4.4273, -5.0039,  1.4893,  ..., 17.1783, -2.3285, -2.5852],
        [ 8.5899, -3.9250, -3.2830,  ..., 14.7427,  3.4850, -2.7827],
        [ 7.1827, -3.2719,  1.7966,  ..., 13.2363, -3.2139, -0.7752]],
       grad_fn=<AddmmBackward0>)
result:
tensor([[-0.8476,  0.7400,  0.9850,  ...,  1.6133, -1.2644, -1.0550],
        [ 0.9865, -1.4902, -0.7422,  ...,  0.7128, -0.5525,  0.2460],
        [-2.3847,  1.5909, -0.9121,  ..., -2.7131,  1.2772,  0.2626]